# Automated Claim Triage: an improved prompt chain

Companion notebook for **lesson 4.4** of the study notes. Three stages turn a free-text auto-insurance claim (FNOL) into a queue decision:

1. **Extract** claim info with the LLM → Gate 1 (Pydantic)
2. **Assess** severity and cost with the LLM → Gate 2 (Pydantic + cost range)
3. **Route** with plain code: the rules are fixed, so no LLM call

Failures retry with the gate's error as feedback; after the retry limit the claim goes to `needs_review`, so nothing is dropped.

**Run the setup cell first.** In Colab it uses Colab's built-in models, no API key needed. If those aren't available on your account, the setup cell says why; then add `GEMINI_API_KEY` in **Secrets** (key icon, left sidebar) with **Notebook access** on.

In [1]:
# Setup: load the shared get_completion() helper, from the repo or (in Colab) from GitHub
import os, sys, urllib.request
for path in ["../../shared", "."]:
    if os.path.exists(os.path.join(path, "llm.py")):
        sys.path.insert(0, path)
        break
else:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/deresegetachew/study-notes/main/udacity-masters-agentic-ai/code/shared/llm.py", "llm.py")
    sys.path.insert(0, ".")

from llm import get_completion, check_setup
check_setup(require=False)   # which LLM will answer, or why none can

✗ No LLM available on this machine.


## 1 · The data contracts

In [2]:
import json, re
from typing import List, Literal
from pydantic import BaseModel, Field, ValidationError

class ClaimInformation(BaseModel):                     # Stage I output
    claim_id: str = Field(..., min_length=2, max_length=10)
    name: str = Field(..., min_length=2, max_length=100)
    vehicle: str = Field(..., min_length=2, max_length=100)
    loss_desc: str = Field(..., min_length=10, max_length=500)
    damage_area: List[Literal["windshield", "front", "rear", "side", "roof", "hood", "door",
                              "bumper", "fender", "quarter panel", "trunk", "glass"]] = Field(..., min_length=1)

class SeverityAssessment(BaseModel):                   # Stage II output
    severity: Literal["Minor", "Moderate", "Major"]
    est_cost: float = Field(..., gt=0)

## 2 · Prompts (from the course exercise)

In [3]:
info_extraction_system_prompt = """
You are an auto insurance claim processing assistant. Extract key information from the
First Notice of Loss (FNOL) report as a JSON object with keys: claim_id, name, vehicle,
loss_desc (concise), damage_area (list; only: windshield, front, rear, side, roof, hood,
door, bumper, fender, quarter panel, trunk, glass). Only respond with the JSON object.
"""

severity_assessment_system_prompt = """
You are an auto insurance damage assessor. Apply these carrier heuristics:
- Minor: small dents, scratches, glass chips ($100-$1,000)
- Moderate: single panel, bumper replacement, door damage ($1,000-$5,000)
- Major: structural, multiple panels, engine/drivetrain, total-loss candidates ($5,000-$50,000)
Respond only with a JSON object: {"severity": "Minor"|"Moderate"|"Major", "est_cost": number}.
"""

## 3 · Gate checks

In [4]:
COST_RANGES = {"Minor": (100, 1_000), "Moderate": (1_000, 5_000), "Major": (5_000, 50_000)}

def gate1(raw: str) -> ClaimInformation:
    """Shape check: valid JSON with the right fields and allowed damage areas."""
    try:
        return ClaimInformation.model_validate_json(raw)
    except ValidationError as e:
        raise ValueError(str(e))

def gate2(raw: str) -> SeverityAssessment:
    """Shape check + logic check: the cost must fit the chosen severity."""
    try:
        sev = SeverityAssessment.model_validate_json(raw)
    except ValidationError as e:
        raise ValueError(str(e))
    low, high = COST_RANGES[sev.severity]
    if not low <= sev.est_cost <= high:
        raise ValueError(f"{sev.severity} damage must cost {low:,}-{high:,} USD, got {sev.est_cost:,.0f}")
    return sev

## 4 · Routing in plain code

In [5]:
GLASS_ONLY = {"windshield", "glass"}

def route(claim: ClaimInformation, sev: SeverityAssessment) -> str:
    """Stage III: the routing rules are fixed, so no LLM call is needed."""
    if sev.severity == "Major":
        return "total_loss"
    if sev.severity == "Moderate":
        return "material_damage"
    if set(claim.damage_area) <= GLASS_ONLY:
        return "glass"
    return "fast_track"

def _claim(areas):
    return ClaimInformation(claim_id="C1", name="Al", vehicle="Car", loss_desc="something happened", damage_area=areas)

for areas, severity, cost in [(["windshield"], "Minor", 150), (["windshield", "door"], "Minor", 300),
                              (["door"], "Moderate", 2000), (["front"], "Major", 15000)]:
    print(f"{severity:8} {str(areas):24} -> {route(_claim(areas), SeverityAssessment(severity=severity, est_cost=cost))}")

Minor    ['windshield']           -> glass
Minor    ['windshield', 'door']   -> fast_track
Moderate ['door']                 -> material_damage
Major    ['front']                -> total_loss


## 5 · The chain: retries with feedback, then human review

In [6]:
class GateFailed(Exception):
    pass

def strip_fences(text: str) -> str:
    """Models often wrap JSON in a Markdown fence; remove it before parsing."""
    return re.sub(r"^\s*`{3}(?:json)?\s*\n|\n?`{3}\s*$", "", text.strip())

def call_with_gate(system_prompt: str, user_content: str, gate, max_attempts: int = 3):
    """One chain step: call the LLM, run the gate, retry with the gate's error as feedback."""
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}]
    for attempt in range(1, max_attempts + 1):
        raw = strip_fences(get_completion(messages=messages, temperature=0))
        try:
            return gate(raw)
        except ValueError as e:
            if attempt == max_attempts:
                raise GateFailed(f"{gate.__name__} failed {max_attempts}x: {e}")
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"Your answer failed validation:\n{e}\nReturn corrected JSON only."}]

def triage(fnol_text: str) -> dict:
    """Never drops a claim: anything that fails a gate goes to human review."""
    try:
        claim = call_with_gate(info_extraction_system_prompt, fnol_text, gate1)
        sev = call_with_gate(severity_assessment_system_prompt, claim.model_dump_json(), gate2)
        return {"claim_id": claim.claim_id, "severity": sev.severity, "est_cost": sev.est_cost,
                "queue": route(claim, sev), "status": "auto"}
    except GateFailed as e:
        return {"claim_id": None, "queue": "needs_review", "status": "human", "reason": str(e)[:120]}

## 6 · Gate checks in action

The gates are plain code, so they can be tested directly with bad LLM-style outputs, no model needed. This is what triggers a retry (with the error as feedback) inside the chain.

In [7]:
def first_problem(error: str) -> str:
    """The first problem a gate reported, on one line: 'field: reason'."""
    lines = [l.strip() for l in error.splitlines() if l.strip() and not l.strip().startswith("For further")]
    if "validation error" not in lines[0]:
        return lines[0]                                  # our own gate message (e.g. cost range)
    count = int(lines[0].split()[0])
    field, reason = (lines[1], lines[2]) if len(lines) > 2 and not lines[1].startswith("Invalid JSON") else ("JSON", lines[1])
    more = f"  (+{count - 1} more)" if count > 1 else ""
    return f"{field}: {reason.split(' [type')[0]}{more}"

bad_outputs = [
    (gate1, "Sure! Here is the claim info you asked for."),                                   # chatty, not JSON
    (gate1, '{"claim_id": "C001", "name": "John Smith"}'),                                    # missing fields
    (gate1, '{"claim_id": "C002", "name": "Sarah Johnson", "vehicle": "2020 Honda Civic", '
            '"loss_desc": "Parked car was hit.", "damage_area": ["taillight"]}'),            # not an allowed area
    (gate2, '{"severity": "Minor", "est_cost": 3000}'),                                       # cost too high for Minor
    (gate2, '{"severity": "Catastrophic", "est_cost": 90000}'),                               # unknown severity
]
for gate, raw in bad_outputs:
    try:
        gate(raw)
        print(f"{gate.__name__} ✅ passed")
    except ValueError as e:
        print(f"{gate.__name__} ❌ {first_problem(str(e))}")

gate1 ❌ JSON: Invalid JSON: expected value at line 1 column 1
gate1 ❌ vehicle: Field required  (+2 more)
gate1 ❌ damage_area.0: Input should be 'windshield', 'front', 'rear', 'side', 'roof', 'hood', 'door', 'bumper', 'fender', 'quarter panel', 'trunk' or 'glass'
gate2 ❌ Minor damage must cost 100-1,000 USD, got 3,000
gate2 ❌ severity: Input should be 'Minor', 'Moderate' or 'Major'


## 7 · Run the chain on the course's sample claims

This calls a real model (Colab built-in, or your key). Compare the queues with the course's solution run: does the hail claim (C004) still come out as Moderate?

In [ ]:
check_setup()   # stops here with instructions if no LLM is available

sample_fnols = {
    "C001": "Claim ID: C001\nCustomer: John Smith\nVehicle: 2018 Toyota Camry\nIncident: While driving on the highway, "
            "a rock hit my windshield and caused a small chip about the size of a quarter. No other damage was observed.",
    "C002": "Claim ID: C002\nCustomer: Sarah Johnson\nVehicle: 2020 Honda Civic\nIncident: I was parked at the grocery store "
            "and returned to find someone had hit my car and dented the rear bumper and taillight. The taillight is broken "
            "and the bumper has a large dent.",
    "C003": "Claim ID: C003\nCustomer: Michael Rodriguez\nVehicle: 2022 Ford F-150\nIncident: I was involved in a serious "
            "collision at an intersection. The front of my truck is severely damaged, including the hood, bumper, radiator, "
            "and engine compartment. The airbags deployed and the vehicle is not drivable.",
    "C004": "Claim ID: C004\nCustomer: Emma Williams\nVehicle: 2019 Subaru Outback\nIncident: My car was damaged in a "
            "hailstorm. There are multiple dents on the hood, roof, and trunk. The side mirrors were also damaged and one "
            "window has a small crack.",
    "C005": "Claim ID: C005\nCustomer: David Brown\nVehicle: 2021 Tesla Model 3\nIncident: Someone keyed my car in the "
            "parking lot. There are deep scratches along both doors on the driver's side.",
}
for claim_id, text in sample_fnols.items():
    r = triage(text)
    detail = f"{r['severity']:8} ${r['est_cost']:>8,.0f}" if r["status"] == "auto" else r["reason"][:60]
    print(f"{claim_id}  {r['queue']:16} {detail}")